In [3]:
!pip install mp-api
apikey = "4Fxu8kCP9i4QlEKWL4gUCKY8WfTujzM8"
import pandas as pd
from mp_api.client import MPRester
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty

fields = [
    "material_id", "formula_pretty", "bulk_modulus", "shear_modulus", 
    "density", "volume", "band_gap", "formation_energy_per_atom"
]

with MPRester(apikey) as mpr:
    docs = mpr.materials.summary.search(has_props=["elasticity"], is_stable=True, fields=fields)
    results = []
    
    for doc in docs:
        b = doc.bulk_modulus.get('vrh') if isinstance(doc.bulk_modulus, dict) else getattr(doc.bulk_modulus, 'vrh', None)
        g = doc.shear_modulus.get('vrh') if isinstance(doc.shear_modulus, dict) else getattr(doc.shear_modulus, 'vrh', None)
        
        if b and g and b > 0 and g > 0:
            k = g / b
            hv = max(0, 2 * (pow(k**2 * g, 0.585)) - 3)
            results.append({
                "id": doc.material_id,
                "formula": doc.formula_pretty,
                "vickers_hardness_gpa": hv,
                "density": doc.density,
                "band_gap": doc.band_gap,
                "formation_energy": doc.formation_energy_per_atom
                })

df = pd.DataFrame(results)

stc = StrToComposition()
df = stc.featurize_dataframe(df, "formula") 

ep_feat = ElementProperty(
    data_source="magpie",
    features=["AtomicWeight", "Number", "Electronegativity", "MeltingT", "CovalentRadius"],
    stats=["mean", "avg_dev", "range"] 
)

df = ep_feat.featurize_dataframe(df, col_id="composition")

df.to_csv("mp_hardness_advanced_v3.csv", index=False)

Retrieving SummaryDoc documents:   0%|          | 0/6087 [00:00<?, ?it/s]

StrToComposition:   0%|          | 0/6087 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/6087 [00:00<?, ?it/s]